In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!pip -q install -U datasets lpips scikit-learn pandas matplotlib seaborn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
dask-

# Cell 1: Configuration

This cell sets up the experiment and defines where its inputs and outputs live. The single-epoch trained victim model checkpoint is read from the existing `GGSS_R_unperturbed_reproduction` directory. Every file generated by this diagnostic is placed under a separate `diagnostics_14th_august` directory. The directories are intentionally kept separate, so that this experiment only inspects the trained model and existing reconstruction results without modifying the original reproduction directory.

The experiment uses 100 CelebA images from class 0 and 100 from class 1, with a fixed seed of 2026. Because the full MLP gradient has 100,729,730 (100.7M) elements, the notebook does not keep the complete gradient population in memory. Instead, for per-element statistics, it samples 100,000 fixed coordinates. As for projections, it constructs a 256-dimensional random projection using 4,096 gradient coordinates per projected dimension (thus accounting for 1M elements in total). 

In [ ]:
# ============================================================
# CELL 1 — CONFIGURATION
# ============================================================

from pathlib import Path
import os
import json
import math
import random
import time
import gc

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image
from torchvision import transforms
from datasets import load_dataset

# ------------------------------------------------------------
# READ-ONLY source
# ------------------------------------------------------------

REPRO_ROOT = Path(
    "/content/drive/MyDrive/GGSS_R_unperturbed_reproduction"
).resolve()

CHECKPOINT_PATH = (
    REPRO_ROOT /
    "model_state" /
    "MLP_1.pth"
)

# ------------------------------------------------------------
# COMPLETELY SEPARATE diagnostic directory
# ------------------------------------------------------------

DIAG_ROOT = Path(
    "/content/drive/MyDrive/diagnostics_14th_august"
).resolve()

DIAG_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

# Everything generated by this notebook goes here.
DATA_DIR       = DIAG_ROOT / "data"
GRAD_DIR       = DIAG_ROOT / "gradient_statistics"
EMBED_DIR      = DIAG_ROOT / "embeddings"
METRIC_DIR     = DIAG_ROOT / "metrics"
PLOT_DIR       = DIAG_ROOT / "plots"
REPORT_DIR     = DIAG_ROOT / "reports"

for p in [
    DATA_DIR,
    GRAD_DIR,
    EMBED_DIR,
    METRIC_DIR,
    PLOT_DIR,
    REPORT_DIR,
]:
    p.mkdir(
        parents=True,
        exist_ok=True,
    )

# ------------------------------------------------------------
# Experiment size
# ------------------------------------------------------------

SEED = 2026

N_CLASS0 = 100
N_CLASS1 = 100

# Fixed number of individual gradient coordinates whose
# per-element statistics we will track.
#
# 100k gives a good statistical sample while avoiding
# allocating ~800 MB for full-coordinate mean/variance.
N_ELEMENT_SAMPLES = 100_000

# Random-projection dimensionality.
#
# This is NOT the full gradient. It is a fixed embedding used
# for population-level pairwise analysis / PCA / clustering.
PROJECTION_DIM = 256

# Number of nearest neighbours to report.
K_NEIGHBORS = 10

COMPUTE_LPIPS = True

DEVICE = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 70)
print("GGSS-R GRADIENT IDENTIFIABILITY DIAGNOSTIC")
print("=" * 70)

print("Reproduction root :", REPRO_ROOT)
print("Checkpoint        :", CHECKPOINT_PATH)
print("Diagnostic root   :", DIAG_ROOT)
print("Device            :", DEVICE)

if torch.cuda.is_available():
    print("GPU               :", torch.cuda.get_device_name(0))

assert CHECKPOINT_PATH.is_file(), (
    f"Checkpoint not found:\n{CHECKPOINT_PATH}"
)

assert not DIAG_ROOT.is_relative_to(REPRO_ROOT), (
    "Safety check failed: diagnostic directory must be "
    "outside GGSS_R_unperturbed_reproduction."
)

print()
print("✓ Checkpoint exists")
print("✓ Diagnostic directory is separate")
print("✓ No diagnostic outputs will be written to the reproduction tree")

GGSS-R GRADIENT IDENTIFIABILITY DIAGNOSTIC
Reproduction root : /content/drive/MyDrive/GGSS_R_unperturbed_reproduction
Checkpoint        : /content/drive/MyDrive/GGSS_R_unperturbed_reproduction/model_state/MLP_1.pth
Diagnostic root   : /content/drive/MyDrive/diagnostics_14th_august
Device            : cuda:0
GPU               : Tesla T4

✓ Checkpoint exists
✓ Diagnostic directory is separate
✓ No diagnostic outputs will be written to the reproduction tree


# Cell 2: Recreate the Paper MLP-3 / Repository MLP_1

This cell rebuilds the victim classifier used for the gradient calculations. The network takes a 3 × 256 × 256 image, flattens it, and passes it through three fully connected layers:

196608 -> 512 -> 128 -> 2

with ReLU activations after the first two layers. The existing checkpoint is then loaded into this exact architecture, and the model is put into evaluation mode.

The parameter count is checked against the expected 100,729,730 parameters. This is an important validation step because the subsequent gradient representation has exactly one coordinate for each model parameter. If the architecture or checkpoint were mismatched, all later gradient-distance measurements would be measuring a different model than the one used in the GGSS-R reproduction.


In [ ]:
# ============================================================
# CELL 2 — RECREATE PAPER-MLP-3 / REPOSITORY MLP_1
# ============================================================

class MLP_1(nn.Module):

    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(
            3 * 256 * 256,
            512,
        )

        self.fc2 = nn.Linear(
            512,
            128,
        )

        self.fc3 = nn.Linear(
            128,
            2,
        )

        self.relu = nn.ReLU()

    def forward(self, x):

        x = x.view(
            x.size(0),
            -1,
        )

        x = self.relu(
            self.fc1(x)
        )

        x = self.relu(
            self.fc2(x)
        )

        x = self.fc3(x)

        return x


model = MLP_1().to(DEVICE)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False,
)

if isinstance(checkpoint, dict):

    if "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]

    elif "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]

    else:
        state_dict = checkpoint

else:
    state_dict = checkpoint

state_dict = {
    k.replace("module.", "", 1): v
    for k, v in state_dict.items()
}

model.load_state_dict(
    state_dict,
    strict=True,
)

model.eval()

num_params = sum(
    p.numel()
    for p in model.parameters()
)

print("Model:")
print(model)

print()
print(f"Total parameters: {num_params:,}")

assert num_params == 100_729_730

print()
print("✓ Checkpoint loaded")
print("✓ Parameter count matches MLP_1")

Model:
MLP_1(
  (fc1): Linear(in_features=196608, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=2, bias=True)
  (relu): ReLU()
)

Total parameters: 100,729,730

✓ Checkpoint loaded
✓ Parameter count matches MLP_1


# Cell 3: Load CelebA into the Diagnostic Directory

This cell loads the CelebA training split and identifies images according to the `Smiling` attribute used as the two-class label. The complete training split contains 162,770 images, with 84,690 labelled class 0 and 78,080 labelled class 1. From these pools, the notebook selects 100 images from each class using the fixed random seed defined earlier.

The class balance is deliberate. The diagnostic is not only asking whether individual images have distinctive gradients; it also asks whether gradient representations contain enough structure to separate two known classes. Using 100 examples from each class gives 200 images and 19,900 unique image pairs for the later pairwise analyses.

The images are downloaded into the diagnostic directory's Hugging Face cache. This keeps the data used by the experiment separate from the original GGSS-R reproduction files.


In [5]:
# ============================================================
# CELL 3 — LOAD CELEBA INTO DIAGNOSTIC DIRECTORY
# ============================================================

print("=" * 70)
print("LOADING CELEBA")
print("=" * 70)

HF_CACHE = DATA_DIR / "hf_cache"

HF_CACHE.mkdir(
    parents=True,
    exist_ok=True,
)

celeba = load_dataset(
    "flwrlabs/celeba",
    split="train",
    cache_dir=str(HF_CACHE),
)

print()
print("Dataset size:", len(celeba))
print("Columns:", celeba.column_names)

assert "image" in celeba.column_names
assert "Smiling" in celeba.column_names

smiling = celeba["Smiling"]

class0_indices = [
    i for i, y in enumerate(smiling)
    if int(y) == 0
]

class1_indices = [
    i for i, y in enumerate(smiling)
    if int(y) == 1
]

print()
print("Class 0:", len(class0_indices))
print("Class 1:", len(class1_indices))

# ------------------------------------------------------------
# Reproducible sampling
# ------------------------------------------------------------

rng = random.Random(SEED)

rng.shuffle(class0_indices)
rng.shuffle(class1_indices)

class0_indices = class0_indices[:N_CLASS0]
class1_indices = class1_indices[:N_CLASS1]

print()
print("Selected:")
print("  class 0:", len(class0_indices))
print("  class 1:", len(class1_indices))

LOADING CELEBA


README.md:   0%|          | 0.00/9.28k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

img_align+identity+attr/train-00000-of-0(…): reconstructing file:   0%|          |  0.00B /  500MB            

...

Generating train split:   0%|          | 0/162770 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/19867 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/19962 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/19 [00:00<?, ?it/s]


Dataset size: 162770
Columns: ['image', 'celeb_id', '5_o_Clock_Shadow', 'Arched_Eyebrows', 'Attractive', 'Bags_Under_Eyes', 'Bald', 'Bangs', 'Big_Lips', 'Big_Nose', 'Black_Hair', 'Blond_Hair', 'Blurry', 'Brown_Hair', 'Bushy_Eyebrows', 'Chubby', 'Double_Chin', 'Eyeglasses', 'Goatee', 'Gray_Hair', 'Heavy_Makeup', 'High_Cheekbones', 'Male', 'Mouth_Slightly_Open', 'Mustache', 'Narrow_Eyes', 'No_Beard', 'Oval_Face', 'Pale_Skin', 'Pointy_Nose', 'Receding_Hairline', 'Rosy_Cheeks', 'Sideburns', 'Smiling', 'Straight_Hair', 'Wavy_Hair', 'Wearing_Earrings', 'Wearing_Hat', 'Wearing_Lipstick', 'Wearing_Necklace', 'Wearing_Necktie', 'Young']

Class 0: 84690
Class 1: 78080

Selected:
  class 0: 100
  class 1: 100


# Cell 4: Preprocessing

This cell defines the image preprocessing pipeline used before an image is passed to the victim MLP. The transformation establishes the exact spatial (256 by 256) resolution expected by the model.

Keeping one preprocessing function for the entire experiment is important because every measured gradient depends on the input tensor. The same transformation is later used for the population images, the existing target image, and the saved GGSS-R reconstructions. This makes the gradient comparisons meaningful. Differences should come from the images rather than from different preprocessing procedures.


In [6]:
# ============================================================
# CELL 4 — PREPROCESSING
# ============================================================

TRANSFORM = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5, 0.5, 0.5),
        (0.5, 0.5, 0.5),
    ),
])

def load_celeb_image(index):

    item = celeba[index]

    image = item["image"].convert("RGB")

    return image


def preprocess_pil(image):

    return TRANSFORM(image)


def tensor_to_rgb01(x):

    # [-1,1] -> [0,1]
    return (
        x.detach()
        .cpu()
        .clamp(-1, 1)
        .add(1)
        .div(2)
    )

# Cell 5: LPIPS

This cell initializes LPIPS when perceptual image comparison is enabled. LPIPS provides a perceptual distance between two images that is different from pixel-wise MSE. It is intended to reflect differences in learned visual features rather than treating every pixel difference equally.

The diagnostic uses LPIPS later when comparing existing GGSS-R reconstructions with the target. Pixel MSE and PSNR remain useful for direct numerical comparison. But LPIPS provides a second image-space measure, because a reconstruction can have a relatively large pixel error while still preserving perceptually meaningful structure.

In [7]:
# ============================================================
# CELL 5 — LPIPS
# ============================================================

if COMPUTE_LPIPS:

    import lpips

    lpips_model = lpips.LPIPS(
        net="alex"
    ).to(DEVICE)

    lpips_model.eval()

    print("✓ LPIPS loaded")

else:

    lpips_model = None

    print("LPIPS disabled")

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 192MB/s]


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
✓ LPIPS loaded


# Cell 6: Fixed Gradient Coordinate Sample

The complete victim gradient contains 100,729,730 elements. Storing every coordinate for 200 population images would require a very large amount of memory. So this cell first records the start and end offsets of every model parameter in the flattened gradient and then selects 100,000 coordinates from the complete gradient.

The selected coordinates are generated once with a fixed random seed and reused for every image. Thus, if \(g(x)\) denotes the full gradient for image \(x\), the same coordinate set \(S\) is used to obtain

$$
g_S(x)=\{g_k(x): k\in S\}.
$$

This sample is used for per-coordinate statistics such as variance and sign entropy. It is not intended to replace the full gradient for the exact target comparison later in the notebook; it is a memory-efficient sample for studying how individual gradient coordinates behave across the population.

The output confirms that 100,000 coordinates were sampled from the 100,729,730-dimensional gradient.


In [8]:
# ============================================================
# CELL 6 — FIXED GRADIENT COORDINATE SAMPLE
# ============================================================

# Build parameter offsets.

PARAM_INFO = []

offset = 0

for name, parameter in model.named_parameters():

    n = parameter.numel()

    PARAM_INFO.append({
        "name": name,
        "shape": tuple(parameter.shape),
        "start": offset,
        "end": offset + n,
        "numel": n,
    })

    offset += n

assert offset == num_params

# ------------------------------------------------------------
# Reproducible random coordinates across the complete
# flattened gradient.
# ------------------------------------------------------------

coord_rng = np.random.default_rng(SEED)

sampled_global_coords = np.sort(
    coord_rng.choice(
        num_params,
        size=min(
            N_ELEMENT_SAMPLES,
            num_params,
        ),
        replace=False,
    )
)

print("Total gradient elements :", num_params)
print("Sampled gradient elements:", len(sampled_global_coords))

Total gradient elements : 100729730
Sampled gradient elements: 100000


# Cell 7: Exact Victim Gradient

This cell defines the function that turns one preprocessed image into its victim-model gradient. The image is passed through the MLP, a cross-entropy loss is computed against `TARGET_CLASS = 0`, and backpropagation produces one gradient tensor for every model parameter.

The gradients are retained in parameter order and later concatenated into a single vector,

$$
g(x)=
abla_W L(W,x),
$$

where \(W\) denotes all victim-model parameters. The function also records the model logits and loss, which are useful for checking that the input is being evaluated by the intended classifier.

The explicit zero-gradient operation before each calculation is important because gradients in PyTorch accumulate by default. Without clearing them, the gradient for one image could contain contributions from previous images and the population comparison would be invalid.


In [9]:
# ============================================================
# CELL 7 — EXACT VICTIM GRADIENT
# ============================================================

criterion = nn.CrossEntropyLoss()

TARGET_CLASS = 0

def compute_gradient(image_tensor):

    model.zero_grad(
        set_to_none=True
    )

    x = image_tensor.unsqueeze(0).to(
        DEVICE,
        non_blocking=True,
    )

    logits = model(x)

    target = torch.tensor(
        [TARGET_CLASS],
        dtype=torch.long,
        device=DEVICE,
    )

    loss = criterion(
        logits,
        target,
    )

    loss.backward()

    gradients = []

    for parameter in model.parameters():

        if parameter.grad is None:

            gradients.append(
                torch.zeros_like(
                    parameter
                )
            )

        else:

            gradients.append(
                parameter.grad.detach()
            )

    return {
        "logits": logits.detach().cpu().numpy()[0],
        "loss": float(loss.detach().cpu()),
        "gradients": gradients,
    }

# Cell 8: Gradient Metrics

This cell defines the numerical quantities used to compare victim gradients. It provides the total gradient norm and layer-wise norms, flattens all parameter gradients into one vector, and computes three forms of similarity:

- Euclidean gradient distance:
$$
d(a,b)=\|a-b\|_2.
$$

- Cosine similarity:
$$\cos(a,b) = \frac{a^\top b}{\|a\|_2\|b\|_2}$$

- Relative gradient distance:
$$
d_{\mathrm{rel}}(a,b)=

\frac{\|a-b\|_2}{\|a\|_2}.
$$

It also computes Pearson correlation after centering each gradient vector. These measures answer slightly different questions. Euclidean distance measures absolute separation, cosine similarity measures directional agreement, and correlation removes the effect of a constant offset in the coordinates.

Having these metrics defined in one place ensures that the population analysis, target analysis, and reconstruction analysis use the same definitions.


In [10]:
# ============================================================
# CELL 8 — GRADIENT METRICS
# ============================================================

def gradient_norms(gradients):

    layer_norms = {}

    total_sq = 0.0

    for info, g in zip(
        PARAM_INFO,
        gradients,
    ):

        value = float(
            torch.linalg.vector_norm(
                g.float()
            ).cpu()
        )

        layer_norms[
            info["name"]
        ] = value

        total_sq += value ** 2

    return (
        math.sqrt(total_sq),
        layer_norms,
    )


def flatten_gradient(
    gradients,
    dtype=torch.float32,
):

    return torch.cat([
        g.reshape(-1).to(dtype)
        for g in gradients
    ])


def gradient_cosine(
    a,
    b,
):

    numerator = torch.dot(
        a,
        b,
    )

    denominator = (
        torch.linalg.vector_norm(a)
        *
        torch.linalg.vector_norm(b)
    )

    return float(
        numerator /
        (denominator + 1e-12)
    )


def gradient_distance(
    a,
    b,
):

    return float(
        torch.linalg.vector_norm(
            a - b
        )
    )


def relative_gradient_distance(
    a,
    b,
):

    return (
        gradient_distance(a, b)
        /
        (
            float(
                torch.linalg.vector_norm(a)
            )
            + 1e-12
        )
    )


def pearson_gradient_correlation(
    a,
    b,
):

    a = a - a.mean()
    b = b - b.mean()

    return float(
        torch.dot(a, b)
        /
        (
            torch.linalg.vector_norm(a)
            *
            torch.linalg.vector_norm(b)
            +
            1e-12
        )
    )

# Cell 9: Memory-Efficient Random Gradient Projection

This cell constructs a second, more compact representation of each full gradient. Each of the 256 projected dimensions samples 4,096 coordinates from the 100,729,730-dimensional gradient, multiplies them by fixed random signs, and averages the result.

For projection dimension \(j\), the representation has the form

$$
z_j(x)=

\frac{1}{|S_j|}
\sum_{k\in S_j}s_{jk}\,g_k(x),
$$

where $S_j$ contains 4,096 randomly selected gradient coordinates and $s_{jk}\in\{-1,+1\}$. The coordinate sets and signs are fixed using a separate deterministic seed.

This is intentionally a very small representation compared with the original gradient. The goal is to test whether useful population-level structure survives when the full gradient is compressed, while avoiding the memory cost of storing all 100,729,730 coordinates for every image.


In [11]:
# ============================================================
# CELL 9 — MEMORY-EFFICIENT RANDOM GRADIENT PROJECTION
# ============================================================

# Number of gradient coordinates used by each projection
# dimension.
#
# This is intentionally sparse. Each projected coordinate is
# an average of a random subset of gradient coordinates.

PROJECTION_COORDS_PER_DIM = 4096

projection_rng = np.random.default_rng(
    SEED + 1
)

projection_indices = []
projection_signs = []

for j in range(PROJECTION_DIM):

    indices = projection_rng.choice(
        num_params,
        size=PROJECTION_COORDS_PER_DIM,
        replace=False,
    )

    signs = projection_rng.choice(
        np.array([-1.0, 1.0]),
        size=PROJECTION_COORDS_PER_DIM,
    )

    projection_indices.append(
        indices
    )

    projection_signs.append(
        signs.astype(np.float32)
    )

projection_indices = np.asarray(
    projection_indices,
    dtype=np.int64,
)

projection_signs = np.asarray(
    projection_signs,
    dtype=np.float32,
)

print(
    "Projection:",
    PROJECTION_DIM,
    "dimensions ×",
    PROJECTION_COORDS_PER_DIM,
    "coordinates"
)

Projection: 256 dimensions × 4096 coordinates


# Cell 10: Project a Full Gradient

This cell defines the function that applies the random projection constructed above to one complete flattened gradient. For each of the 256 output dimensions, it gathers the corresponding 4,096 coordinates, applies the fixed random signs, and takes their mean.

The same projection is applied to every image. Therefore, two images are compared in exactly the same 256-dimensional coordinate system. This representation is used for population-wide pairwise analysis, PCA, clustering, and nearest-neighbour searches.

An important caveat is that this projection is a diagnostic representation, not the exact 100-million-dimensional gradient. Results based on it should therefore be interpreted as evidence about the compressed gradient representation, while the later target analysis recomputes full gradients when an exact comparison is required.


In [12]:
# ============================================================
# CELL 10 — PROJECT A FULL GRADIENT
# ============================================================

def project_gradient(flat_gradient):

    output = np.empty(
        PROJECTION_DIM,
        dtype=np.float32,
    )

    for j in range(PROJECTION_DIM):

        coords = projection_indices[j]

        values = (
            flat_gradient[coords]
            .numpy()
        )

        output[j] = np.mean(
            values *
            projection_signs[j]
        )

    return output

# Cell 11: Image-Space Metrics

This cell defines the image-space measures used later to compare reconstructions or population images with the target. Pixel MSE measures the average squared difference between corresponding normalized pixels,

$$
\mathrm{MSE}(x,y)=

\frac{1}{N}\sum_{p=1}^{N}(x_p-y_p)^2.
$$

PSNR is derived from MSE as

$$
\mathrm{PSNR}=10\log_{10}\left(
\frac{1}{\mathrm{MSE}}
\right),
$$

because the images are represented in the normalized $[0,1]$ range. 

LPIPS is used as a perceptual distance.

These metrics are kept separate from the gradient metrics because image similarity and gradient similarity do NOT necessarily measure the same property. One of the main questions of this notebook is whether closeness in gradient space is actually associated with closeness in image space. 


In [13]:
# ============================================================
# CELL 11 — IMAGE-SPACE METRICS
# ============================================================

def pixel_mse(a, b):

    return float(
        torch.mean(
            (a - b) ** 2
        )
    )


def psnr_from_mse(
    mse,
):

    if mse <= 0:
        return float("inf")

    return float(
        10.0 *
        math.log10(
            1.0 / mse
        )
    )


def compute_lpips(
    a,
    b,
):

    if lpips_model is None:
        return float("nan")

    # LPIPS expects [-1,1]
    with torch.no_grad():

        value = lpips_model(
            a.unsqueeze(0).to(DEVICE),
            b.unsqueeze(0).to(DEVICE),
        )

    return float(
        value.squeeze().cpu()
    )

# Cell 12: Compute Gradient Dataset

This cell computes the victim gradient for all 200 selected CelebA images: 100 from class 0 and 100 from class 1. For each image, it records the loss, logits, total gradient norm, and layer-wise gradient norms. It also stores two reduced gradient representations: the 100,000 fixed coordinates and the 256-dimensional random projection.

The complete flattened gradient is deliberately discarded after these quantities are extracted. This is what makes it possible to process the entire population on the available Tesla T4 without retaining 200 full 100-million-dimensional gradients.

The run processes all 200 images successfully in about 1.33 minutes, reaching roughly 2.5 images per second. This successful completion is important as all later population statistics are based on the same 200-image gradient dataset.


In [14]:
# ============================================================
# CELL 12 — COMPUTE GRADIENT DATASET
# ============================================================

records = []

gradient_embeddings = []

sampled_coordinate_matrix = []

start_time = time.time()

all_indices = (
    [(i, 0) for i in class0_indices]
    +
    [(i, 1) for i in class1_indices]
)

print("=" * 70)
print("COMPUTING GRADIENT POPULATION")
print("=" * 70)

print()
print("Images:", len(all_indices))
print("This may take a while on a T4.")
print()

for counter, (dataset_index, class_label) in enumerate(
    all_indices,
    start=1,
):

    image = load_celeb_image(
        dataset_index
    )

    x = preprocess_pil(
        image
    )

    result = compute_gradient(
        x
    )

    gradients = result["gradients"]

    flat = flatten_gradient(
        gradients
    ).cpu()

    total_norm, layer_norms = (
        gradient_norms(
            gradients
        )
    )

    # --------------------------------------------------------
    # Fixed coordinate sample
    # --------------------------------------------------------

    sampled_values = (
        flat[
            sampled_global_coords
        ]
        .numpy()
        .astype(np.float32)
    )

    sampled_coordinate_matrix.append(
        sampled_values
    )

    # --------------------------------------------------------
    # Compact gradient projection
    # --------------------------------------------------------

    embedding = project_gradient(
        flat
    )

    gradient_embeddings.append(
        embedding
    )

    # --------------------------------------------------------
    # Store basic metadata
    # --------------------------------------------------------

    records.append({
        "row": counter - 1,
        "dataset_index": dataset_index,
        "class": class_label,
        "loss": result["loss"],
        "logit_0": result["logits"][0],
        "logit_1": result["logits"][1],
        "gradient_norm": total_norm,
        **{
            f"gradnorm_{k}": v
            for k, v in layer_norms.items()
        },
    })

    # --------------------------------------------------------
    # Free GPU memory
    # --------------------------------------------------------

    del gradients
    del flat
    del result
    del x
    del image

    model.zero_grad(
        set_to_none=True
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if counter % 10 == 0:

        elapsed = (
            time.time()
            - start_time
        )

        rate = (
            counter /
            elapsed
        )

        remaining = (
            len(all_indices)
            - counter
        ) / rate

        print(
            f"{counter:4d}/{len(all_indices)} "
            f"| {rate:.2f} img/s "
            f"| ETA {remaining/60:.1f} min"
        )

print()
print(
    "Elapsed:",
    (time.time() - start_time) / 60,
    "minutes"
)

print()
print("✓ Gradient population computed")

COMPUTING GRADIENT POPULATION

Images: 200
This may take a while on a T4.

  10/200 | 2.09 img/s | ETA 1.5 min
  20/200 | 2.24 img/s | ETA 1.3 min
  30/200 | 2.37 img/s | ETA 1.2 min
  40/200 | 2.43 img/s | ETA 1.1 min
  50/200 | 2.38 img/s | ETA 1.0 min
  60/200 | 2.43 img/s | ETA 1.0 min
  70/200 | 2.46 img/s | ETA 0.9 min
  80/200 | 2.44 img/s | ETA 0.8 min
  90/200 | 2.46 img/s | ETA 0.7 min
 100/200 | 2.48 img/s | ETA 0.7 min
 110/200 | 2.48 img/s | ETA 0.6 min
 120/200 | 2.47 img/s | ETA 0.5 min
 130/200 | 2.48 img/s | ETA 0.5 min
 140/200 | 2.50 img/s | ETA 0.4 min
 150/200 | 2.48 img/s | ETA 0.3 min
 160/200 | 2.49 img/s | ETA 0.3 min
 170/200 | 2.50 img/s | ETA 0.2 min
 180/200 | 2.49 img/s | ETA 0.1 min
 190/200 | 2.49 img/s | ETA 0.1 min
 200/200 | 2.50 img/s | ETA 0.0 min

Elapsed: 1.334246023495992 minutes

✓ Gradient population computed


### Interpretation

The population computation completed successfully for all 200 images. Every later population statistic is based on the same fixed set of 200 images and the same victim model.


# Cell 13: Save Compact Data

This cell converts the collected population records and reduced gradient representations into persistent files under the diagnostic directory. The metadata table stores the image indices, class labels, losses, logits, and gradient norms. The embedding arrays store the compact gradient representations. The sampled-coordinate matrix stores the 100,000 selected gradient coordinates for each image.

Only compact representations are saved rather than the complete gradient for every image. This keeps the diagnostic reproducible while avoiding an unnecessarily large intermediate dataset.

The output confirms that the diagnostic data were written to `diagnostics_14th_august`, separate from the original GGSS-R reproduction tree.


In [15]:
# ============================================================
# CELL 13 — SAVE COMPACT DATA
# ============================================================

records_df = pd.DataFrame(
    records
)

embeddings = np.asarray(
    gradient_embeddings,
    dtype=np.float32,
)

coordinate_matrix = np.asarray(
    sampled_coordinate_matrix,
    dtype=np.float32,
)

records_df.to_csv(
    METRIC_DIR /
    "gradient_population_metadata.csv",
    index=False,
)

np.save(
    EMBED_DIR /
    "gradient_random_projection.npy",
    embeddings,
)

np.save(
    GRAD_DIR /
    "sampled_gradient_coordinates.npy",
    coordinate_matrix,
)

np.save(
    GRAD_DIR /
    "sampled_global_coordinate_indices.npy",
    sampled_global_coords,
)

np.save(
    EMBED_DIR /
    "projection_indices.npy",
    projection_indices,
)

np.save(
    EMBED_DIR /
    "projection_signs.npy",
    projection_signs,
)

print("Saved diagnostic data to:")
print(DIAG_ROOT)

Saved diagnostic data to:
/content/drive/MyDrive/diagnostics_14th_august


# Cell 14: Per-Element Gradient Statistics

This cell examines how individual sampled gradient coordinates vary across the 200-image population. For each of the 100,000 sampled coordinates it computes the mean, variance, standard deviation, absolute mean, coefficient of variation, fraction of positive values, and sign entropy.

For a coordinate whose gradient is positive with probability \(p\), sign entropy is

$$
H(p)=
-p\log_2 p-(1-p)\log_2(1-p).
$$

A value near 0 means the sign is almost constant across the population, while a value near 1 bit means positive and negative signs occur with roughly equal frequency.

The observed median variance is low at only $(6.28 \times 10^{-8})$, the median sign entropy is about 0.242 bits, and only 10.098% of sampled coordinates have sign entropy above 0.9 bits. Thus, most sampled coordinates do not have highly variable signs across this population, although a small subset does. This provides evidence that gradient variability is generally low across the 200 image population.

As most of the observed 100,000 gradient coordinates have almost miniscule variance and small sign entropy, it is likely that the gradients are failing to uniquely identify an image.


In [16]:
# ============================================================
# CELL 14 — PER-ELEMENT GRADIENT STATISTICS
# ============================================================

X = coordinate_matrix

means = np.mean(
    X,
    axis=0,
)

variances = np.var(
    X,
    axis=0,
)

stds = np.sqrt(
    variances
)

abs_means = np.abs(
    means
)

cv = (
    stds /
    (abs_means + 1e-12)
)

positive_fraction = np.mean(
    X > 0,
    axis=0,
)

sign_entropy = -(
    positive_fraction *
    np.log2(
        positive_fraction + 1e-12
    )
    +
    (1 - positive_fraction) *
    np.log2(
        1 - positive_fraction + 1e-12
    )
)

element_stats = pd.DataFrame({
    "global_gradient_index":
        sampled_global_coords,

    "mean":
        means,

    "variance":
        variances,

    "std":
        stds,

    "abs_mean":
        abs_means,

    "coefficient_of_variation":
        cv,

    "positive_fraction":
        positive_fraction,

    "sign_entropy_bits":
        sign_entropy,
})

element_stats.to_csv(
    GRAD_DIR /
    "per_element_gradient_statistics.csv",
    index=False,
)

print("=" * 70)
print("PER-ELEMENT GRADIENT STATISTICS")
print("=" * 70)

print()
print(
    "Median variance:",
    np.median(variances)
)

print(
    "Median sign entropy:",
    np.median(sign_entropy)
)

print(
    "Fraction of sampled coordinates with",
    "high sign entropy (>0.9 bits):",
    np.mean(sign_entropy > 0.9)
)

print()
print("✓ Per-element statistics saved")

PER-ELEMENT GRADIENT STATISTICS

Median variance: 6.284814e-08
Median sign entropy: 0.24229218907952943
Fraction of sampled coordinates with high sign entropy (>0.9 bits): 0.10098

✓ Per-element statistics saved


### Interpretation

The low median sign entropy (0.242 bits) indicates that most sampled gradient coordinates have a sign that is relatively stable across the 200 images. Only 10.1% of the sampled coordinates have entropy above 0.9 bits, meaning that only a small subset changes sign frequently across the population. The median variance is also extremely low at $(6.28 \times 10^{-8})$, showing that the variability between gradients is almost negligible.

This suggests that the gradients may not be able to uniquely discernible, and thus cannot be traced back to their unique corresponding images. This would explain the failure in reconstruction in the GGSSR reproduction experiment, where despite the gradient distance between the target image and the predicted images being effectively minimized, the image-space reconstruction quality remained catastrophic, with high MSE, high LPIPS and low PSNR.


# Cell 15: Pairwise Gradient Similarity

This cell computes three pairwise relationships between the 256-dimensional gradient embeddings for all 200 images: cosine similarity, Euclidean distance, and correlation. The result is a 200 × 200 matrix for each measure.

The matrices provide the basis for the population-level analysis that follows. In particular, they allow the experiment to ask whether images from the same CelebA class tend to occupy similar locations in gradient space, whether different classes separate, and which images are nearest neighbours under the gradient representation.

The output only confirms that the matrices were computed; the numerical interpretation is deferred to the next cells where the matrices are grouped and summarized.


In [17]:
# ============================================================
# CELL 15 — PAIRWISE GRADIENT SIMILARITY
# ============================================================

from sklearn.metrics import pairwise_distances

# Normalize embeddings.
embedding_norms = np.linalg.norm(
    embeddings,
    axis=1,
    keepdims=True,
)

normalized_embeddings = (
    embeddings /
    (embedding_norms + 1e-12)
)

cosine_matrix = (
    normalized_embeddings
    @
    normalized_embeddings.T
)

euclidean_matrix = pairwise_distances(
    embeddings,
    metric="euclidean",
)

# Correlation is equivalent to cosine after
# centering each vector.
centered = (
    embeddings -
    embeddings.mean(
        axis=1,
        keepdims=True,
    )
)

centered_norms = np.linalg.norm(
    centered,
    axis=1,
    keepdims=True,
)

normalized_centered = (
    centered /
    (centered_norms + 1e-12)
)

correlation_matrix = (
    normalized_centered
    @
    normalized_centered.T
)

np.save(
    METRIC_DIR /
    "gradient_cosine_matrix.npy",
    cosine_matrix,
)

np.save(
    METRIC_DIR /
    "gradient_euclidean_matrix.npy",
    euclidean_matrix,
)

np.save(
    METRIC_DIR /
    "gradient_correlation_matrix.npy",
    correlation_matrix,
)

print("✓ Pairwise matrices computed")

✓ Pairwise matrices computed


# Cell 16: Class Separation

This cell separates all unique image pairs into same-class and different-class groups and compares their gradient cosine similarities and Euclidean distances. With 100 images per class, there are 9,900 (i.e. 100 * 100 - 100) same-class pairs and 10,000 (i.e. 100 * 100) different-class pairs.

The measured means show a modest class effect in the projected gradient space:

- Same-class pairs: mean cosine \(0.5165\), median \(0.5336\), mean distance \(0.001433\).
- Different-class pairs: mean cosine \(0.4819\), median \(0.4881\), mean distance \(0.001961\).

Thus, images sharing the `Smiling` label are somewhat closer in the gradient embedding than images from different classes, as shown by the slightly higher mean cosine and lower mean distance. However, the distributions overlap substantially, so the result should not be interpreted as strong class separation. It indicates that the victim gradients contain some class-related structure, but the embedding is far from being determined solely by the class label.


In [18]:
# ============================================================
# CELL 16 — CLASS SEPARATION
# ============================================================

classes = records_df["class"].to_numpy()

same_class_cosines = []
different_class_cosines = []

same_class_distances = []
different_class_distances = []

n = len(classes)

for i in range(n):

    for j in range(i + 1, n):

        if classes[i] == classes[j]:

            same_class_cosines.append(
                cosine_matrix[i, j]
            )

            same_class_distances.append(
                euclidean_matrix[i, j]
            )

        else:

            different_class_cosines.append(
                cosine_matrix[i, j]
            )

            different_class_distances.append(
                euclidean_matrix[i, j]
            )

class_summary = pd.DataFrame([
    {
        "comparison": "same_class",
        "n_pairs": len(same_class_cosines),
        "mean_cosine": np.mean(same_class_cosines),
        "median_cosine": np.median(same_class_cosines),
        "mean_distance": np.mean(same_class_distances),
        "median_distance": np.median(same_class_distances),
    },
    {
        "comparison": "different_class",
        "n_pairs": len(different_class_cosines),
        "mean_cosine": np.mean(different_class_cosines),
        "median_cosine": np.median(different_class_cosines),
        "mean_distance": np.mean(different_class_distances),
        "median_distance": np.median(different_class_distances),
    },
])

print(class_summary)

class_summary.to_csv(
    METRIC_DIR /
    "class_gradient_separation.csv",
    index=False,
)

        comparison  n_pairs  mean_cosine  median_cosine  mean_distance  \
0       same_class     9900     0.516545       0.533580       0.001433   
1  different_class    10000     0.481854       0.488063       0.001961   

   median_distance  
0         0.001450  
1         0.001922  


### Interpretation

The class comparison shows that the projected gradients are somewhat more similar within the same `Smiling` class than across classes. The difference in mean cosine is about 0.035, while the mean Euclidean distance differs by about 0.00053.

The separation is therefore real in the sampled embedding but modest. There is substantial overlap between the two groups, so the gradient representation cannot be described as a simple class code. This is consistent with the later nearest-neighbour results, where some nearest neighbours belong to the opposite class.


# Cell 17: PCA of Gradient Space

This cell applies principal component analysis to the 256-dimensional gradient embeddings. PCA finds directions that explain the largest amount of variation across the 200 images and provides a lower-dimensional description of the population.

The cumulative variance results are:

- 2 components explain 50%.
- 5 components explain 80%.
- 9 components explain 90%.
- 16 components explain 95%.
- 39 components explain 99%.

The fact that only 16 components explain 95% of the variance means that the 256-dimensional projected population has substantial redundancy. This does not mean that the original 100-million-dimensional gradients are intrinsically 16-dimensional; PCA is being applied after the specific random projection used in this diagnostic. The result instead shows that, within this compact representation and this sampled population, much of the observed variation lies in a relatively low-dimensional subspace.


In [19]:
# ============================================================
# CELL 17 — PCA OF GRADIENT SPACE
# ============================================================

from sklearn.decomposition import PCA

n_components = min(
    50,
    embeddings.shape[0] - 1,
)

pca = PCA(
    n_components=n_components,
    random_state=SEED,
)

embedding_pca = pca.fit_transform(
    embeddings
)

explained = pca.explained_variance_ratio_

pca_df = pd.DataFrame({
    "component": np.arange(
        1,
        n_components + 1,
    ),

    "explained_variance_ratio":
        explained,

    "cumulative_explained_variance":
        np.cumsum(explained),
})

pca_df.to_csv(
    EMBED_DIR /
    "gradient_pca_variance.csv",
    index=False,
)

np.save(
    EMBED_DIR /
    "gradient_pca_coordinates.npy",
    embedding_pca,
)

print("=" * 70)
print("PCA")
print("=" * 70)

for threshold in [
    0.50,
    0.80,
    0.90,
    0.95,
    0.99,
]:

    indices = np.where(
        np.cumsum(explained)
        >= threshold
    )[0]

    if len(indices):

        print(
            f"{threshold*100:.0f}% variance:",
            indices[0] + 1,
            "components"
        )

    else:

        print(
            f"{threshold*100:.0f}% variance:",
            ">",
            n_components
        )

PCA
50% variance: 2 components
80% variance: 5 components
90% variance: 9 components
95% variance: 16 components
99% variance: 39 components


### Interpretation

The PCA result shows that the projected population has a strong low-dimensional structure: 95% of its variance is captured by 16 principal components and 99% by 39. This means that many of the 256 projected coordinates are correlated or redundant across this particular population. The gradient vectors across images vary along a few primary directional axes rather than scattering uniformly across dimensions.

Although the PCA is applied on the 256-dimension gradient projections (which were built with 4096 original coordinates averaged into every projected dimension, thus in total accounting for 1M gradient coordinates), and not on the original 100M gradient coordinates, it does provide insight into the small subspace dimensionality of the gradients.


# Cell 18: Clustering

This cell applies K-means clustering to the PCA representation for several candidate numbers of clusters and evaluates each solution with the silhouette score. The silhouette score measures how well each point fits within its assigned cluster relative to neighbouring clusters; larger values indicate cleaner separation. Scores can range from -1 to +1; scores closer to +1 signify optimal clustering.

The best score among the tested values occurs at \(k=2\), with a silhouette score of 0.4035. The other tested values are lower: 0.3632 for \(k=3\), 0.3744 for \(k=4\), 0.3622 for \(k=5\), 0.3799 for \(k=8\), and 0.3620 for \(k=10\).

The \(k=2\) result is consistent with the presence of two broad structures in the projected gradient population, which is also compatible with the modest class separation observed in Cell 16. However, K-means is unsupervised and the cluster labels are not compared directly with the CelebA class labels here. Therefore, the result should be treated as evidence of structure in gradient space, not as proof that the gradient representation cleanly encodes the `Smiling` attribute.


In [20]:
# ============================================================
# CELL 18 — CLUSTERING
# ============================================================

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# We deliberately test several cluster counts.
cluster_results = []

for k in [
    2,
    3,
    4,
    5,
    8,
    10,
]:

    if k >= len(embeddings):
        continue

    km = KMeans(
        n_clusters=k,
        random_state=SEED,
        n_init=20,
    )

    labels = km.fit_predict(
        embedding_pca[
            :, :
            min(20, embedding_pca.shape[1])
        ]
    )

    silhouette = silhouette_score(
        embedding_pca,
        labels,
    )

    cluster_results.append({
        "k": k,
        "silhouette": silhouette,
    })

cluster_df = pd.DataFrame(
    cluster_results
)

cluster_df.to_csv(
    METRIC_DIR /
    "gradient_clustering.csv",
    index=False,
)

print(cluster_df)

    k  silhouette
0   2    0.403524
1   3    0.363236
2   4    0.374352
3   5    0.362221
4   8    0.379921
5  10    0.362015


### Interpretation

The best silhouette score is obtained with two clusters (0.4035), while all other tested cluster counts give lower scores. This indicates that two broad groups provide the cleanest K-means partition among the tested choices.

However, the score is moderate rather than close to 1, so the clusters are not cleanly separated. Also, the clustering is unsupervised and the code doesn't compare cluster assignments with the actual `Smiling` labels. The safe conclusion is that the gradient population has broad internal structure, not that K-means has recovered the two CelebA classes exactly.


# Cell 19: Gradient-Space Nearest Neighbours

This cell finds the ten nearest neighbours of every image using Euclidean distance in the 256-dimensional gradient embedding. The query image itself is excluded, and the neighbour's gradient cosine similarity and correlation are recorded alongside the distance.

The first two queries illustrate an important point. For dataset index 95033 (class 0), the closest neighbour is dataset index 46519 from class 1, with embedding distance \(0.000405\) and cosine similarity 0.9713. For dataset index 111591 (class 0), the closest neighbour is dataset index 1586 from class 0, with distance \(0.000267\) and cosine similarity 0.8963.

Therefore, nearest neighbours in gradient space are not restricted to the same semantic class. A class-0 image can have a class-1 image as its closest gradient neighbour. This supports the idea that the gradient representation carries at least some level of image-specific variation beyond the binary class label. However, it also shows that gradient proximity should not automatically be equated with semantic image similarity.


In [21]:
# ============================================================
# CELL 19 — GRADIENT-SPACE NEAREST NEIGHBOURS
# ============================================================

neighbor_rows = []

for i in range(n):

    distances = euclidean_matrix[i].copy()

    distances[i] = np.inf

    neighbors = np.argsort(
        distances
    )[:K_NEIGHBORS]

    for rank, j in enumerate(
        neighbors,
        start=1,
    ):

        neighbor_rows.append({
            "query_row": i,
            "query_dataset_index":
                records_df.loc[
                    i,
                    "dataset_index"
                ],
            "query_class":
                int(classes[i]),

            "neighbor_rank":
                rank,

            "neighbor_row":
                int(j),

            "neighbor_dataset_index":
                records_df.loc[
                    j,
                    "dataset_index"
                ],

            "neighbor_class":
                int(classes[j]),

            "gradient_embedding_distance":
                float(
                    euclidean_matrix[i, j]
                ),

            "gradient_cosine":
                float(
                    cosine_matrix[i, j]
                ),

            "gradient_correlation":
                float(
                    correlation_matrix[i, j]
                ),
        })

neighbors_df = pd.DataFrame(
    neighbor_rows
)

neighbors_df.to_csv(
    METRIC_DIR /
    "gradient_nearest_neighbors.csv",
    index=False,
)

print(
    neighbors_df.head(20)
)

    query_row  query_dataset_index  query_class  neighbor_rank  neighbor_row  \
0           0                95033            0              1           147   
1           0                95033            0              2            94   
2           0                95033            0              3           140   
3           0                95033            0              4            29   
4           0                95033            0              5           172   
5           0                95033            0              6            83   
6           0                95033            0              7            44   
7           0                95033            0              8            53   
8           0                95033            0              9            23   
9           0                95033            0             10            49   
10          1               111591            0              1            65   
11          1               111591      

### Interpretation

The nearest-neighbour results show that high gradient similarity does not force two images to have the same class label. The closest neighbour for query row 0 is a class-1 image even though the query itself is class 0, and its cosine similarity is 0.9713. Another class-0 query has a class-0 nearest neighbour with a lower cosine of 0.8963.

This is useful evidence that the gradient representation contains finer information than the binary class label. At the same time, it warns against interpreting a gradient-space nearest neighbour as a semantic nearest neighbour. The two notions of similarity are related, but they are not identical.


# Cell 20: Compute Image Metrics for the Population

This cell loads and preprocesses the same 200 population images used for the gradient analysis and stores their tensors together. The resulting tensor has shape `(200, 3, 256, 256)`.

Keeping the image tensors in the same population order as the gradient records is important because the next cells compare the two spaces pair by pair. Each row therefore has both an image-space representation and a gradient-space representation for the same underlying CelebA image.


In [22]:
# ============================================================
# CELL 20 — COMPUTE IMAGE METRICS FOR THE POPULATION
# ============================================================

image_tensors = []

for dataset_index, class_label in all_indices:

    image = load_celeb_image(
        dataset_index
    )

    x = preprocess_pil(
        image
    )

    image_tensors.append(
        x.cpu()
    )

image_tensors = torch.stack(
    image_tensors
)

print(
    "Image tensor shape:",
    tuple(image_tensors.shape)
)

Image tensor shape: (200, 3, 256, 256)


# Cell 21: Image-Space Pairwise Metrics

This cell computes pixel-wise MSE between every pair of the 200 population images and stores the resulting 200 × 200 matrix.

The purpose is to create an image-space reference against which the gradient-space relationships can be tested. If images that are close in gradient space are also consistently close in pixel space, that would suggest that gradient proximity reflects image similarity. If the two relationships differ, then the gradient space is encoding information that is not simply a proxy for pixel similarity.

Only pixel MSE is used for this population-wide matrix because computing perceptual metrics for all 19,900 pairs would be much more expensive. PSNR is derived later where needed.


In [23]:
# ============================================================
# CELL 21 — IMAGE-SPACE PAIRWISE METRICS
# ============================================================

image_flat = image_tensors.reshape(
    len(image_tensors),
    -1,
).numpy()

image_pairwise_mse = np.empty(
    (n, n),
    dtype=np.float32,
)

for i in range(n):

    diff = (
        image_flat
        -
        image_flat[i:i+1]
    )

    image_pairwise_mse[i] = np.mean(
        diff ** 2,
        axis=1,
    )

np.save(
    METRIC_DIR /
    "image_pairwise_mse.npy",
    image_pairwise_mse,
)

print("✓ Pixel pairwise MSE computed")

✓ Pixel pairwise MSE computed


# Cell 22: Gradient Distance vs Image Distance

This cell combines the pairwise gradient and image measurements into one table containing all 19,900 unique image pairs. For every pair it records gradient cosine similarity, gradient Euclidean distance, gradient correlation, pixel MSE, and pixel PSNR, together with the class labels.

The resulting population has mean gradient cosine 0.4991 and mean gradient distance 0.001698, while mean pixel MSE is 0.6658. These values describe the overall scale of the sampled population, but the more important question is whether the two spaces vary together. That question is tested explicitly in the next cell using correlation coefficients.

The table is also the source for the scatter plots generated later.


In [24]:
# ============================================================
# CELL 22 — GRADIENT DISTANCE vs IMAGE DISTANCE
# ============================================================

relationship_rows = []

for i in range(n):

    for j in range(i + 1, n):

        relationship_rows.append({
            "i": i,
            "j": j,

            "class_i":
                int(classes[i]),

            "class_j":
                int(classes[j]),

            "gradient_cosine":
                float(
                    cosine_matrix[i, j]
                ),

            "gradient_distance":
                float(
                    euclidean_matrix[i, j]
                ),

            "gradient_correlation":
                float(
                    correlation_matrix[i, j]
                ),

            "pixel_mse":
                float(
                    image_pairwise_mse[i, j]
                ),

            "pixel_psnr":
                psnr_from_mse(
                    float(
                        image_pairwise_mse[i, j]
                    )
                ),
        })

relationship_df = pd.DataFrame(
    relationship_rows
)

relationship_df.to_csv(
    METRIC_DIR /
    "gradient_vs_image_relationship.csv",
    index=False,
)

print(
    relationship_df.describe()
)

                  i             j       class_i       class_j  \
count  19900.000000  19900.000000  19900.000000  19900.000000   
mean      66.000000    133.000000      0.248744      0.751256   
std       47.022453     47.022453      0.432296      0.432296   
min        0.000000      1.000000      0.000000      0.000000   
25%       26.000000    100.000000      0.000000      1.000000   
50%       58.000000    141.000000      0.000000      1.000000   
75%       99.000000    173.000000      0.000000      1.000000   
max      198.000000    199.000000      1.000000      1.000000   

       gradient_cosine  gradient_distance  gradient_correlation     pixel_mse  \
count     19900.000000       19900.000000          19900.000000  19900.000000   
mean          0.499113           0.001698              0.499689      0.665775   
std           0.260139           0.000878              0.259919      0.339370   
min          -0.202255           0.000002             -0.198996      0.073414   
25%      

### Interpretation

The descriptive statistics establish the scale of the pairwise dataset before the correlation test. There are 19,900 unique pairs, with mean projected-gradient cosine 0.4991 and mean pixel MSE 0.6658.

The wide ranges are important: gradient cosine extends from -0.2023 to 0.9916, while pixel MSE ranges from 0.0734 to 2.4312. This means the population contains both highly similar and substantially different pairs in both spaces, giving the correlation analysis enough variation to test whether the two notions of similarity track one another.


# Cell 23: Gradient / Image Correlations

This cell directly tests whether gradient similarity is associated with image similarity across the 19,900 population pairs. Spearman correlation is used because the question is primarily about whether the ranking of pairs is related, rather than whether the relationship is strictly linear.

The results are:

- Gradient distance vs. pixel MSE: $\rho=0.1250$, $p=4.50 \times 10^{-70}$.
- Gradient cosine vs. pixel MSE: $\rho=-0.6348$, $p \approx 0$.
- Gradient distance vs. $(1-cosine)$: $\rho=0.0928$, $p=2.59 \times 10^{-39}$.

The negative cosine/MSE correlation is substantial: pairs with higher gradient cosine similarity tend to have lower pixel MSE. In contrast, Euclidean gradient distance has only a weak positive relationship with pixel MSE despite the very small p-value. The p-values are extremely small partly because there are 19,900 pairs, so statistical significance should not be confused with a large effect size.

The third result is also useful as a consistency check. Euclidean distance and cosine dissimilarity are related measures, but here their rank correlation is only 0.093, showing that they capture different aspects of the projected gradient geometry. This is one reason the later target analysis reports several gradient metrics instead of relying on distance alone.


In [25]:
# ============================================================
# CELL 23 — GRADIENT / IMAGE CORRELATIONS
# ============================================================

from scipy.stats import spearmanr, pearsonr

gradient_distance_values = (
    relationship_df[
        "gradient_distance"
    ].to_numpy()
)

gradient_cosine_values = (
    relationship_df[
        "gradient_cosine"
    ].to_numpy()
)

pixel_mse_values = (
    relationship_df[
        "pixel_mse"
    ].to_numpy()
)

print("=" * 70)
print("GRADIENT ↔ IMAGE RELATIONSHIP")
print("=" * 70)

print()
print(
    "Spearman:"
)

rho, p = spearmanr(
    gradient_distance_values,
    pixel_mse_values,
)

print(
    "  gradient distance vs pixel MSE:",
    rho,
    "p =",
    p,
)

rho, p = spearmanr(
    gradient_cosine_values,
    pixel_mse_values,
)

print(
    "  gradient cosine vs pixel MSE:",
    rho,
    "p =",
    p,
)

rho, p = spearmanr(
    gradient_distance_values,
    1.0 - gradient_cosine_values,
)

print(
    "  gradient distance vs (1-cosine):",
    rho,
    "p =",
    p,
)

GRADIENT ↔ IMAGE RELATIONSHIP

Spearman:
  gradient distance vs pixel MSE: 0.12495957132009494 p = 4.501087354741963e-70
  gradient cosine vs pixel MSE: -0.6348428417996964 p = 0.0
  gradient distance vs (1-cosine): 0.09279676512210577 p = 2.594091431816901e-39


### Interpretation

The strongest result here is the Spearman correlation of -0.6348 between gradient cosine and pixel MSE. Higher cosine similarity generally corresponds to lower pixel MSE, so gradient direction carries substantial information about image similarity in this population.

The Euclidean gradient-distance relationship is much weaker $(rho=0.1250)$, despite its extremely small p-value. With 19,900 pairs, even a small association can be statistically significant. The effect size therefore matters more than the p-value for interpreting this diagnostic.

The difference between the distance and cosine results also shows why a single gradient metric would be insufficient. Cosine focuses on direction, while Euclidean distance depends on both direction and magnitude. In this experiment, directional agreement is much more strongly aligned with pixel similarity than the raw projected-gradient distance.


# Cell 24: Optional Existing Target

This cell checks whether the original GGSS-R reproduction already contains the target image used in the reconstruction experiment. When the file exists, it is loaded with the same preprocessing as the population images, and its exact victim gradient is recomputed.

The output confirms that the existing target was found. The victim model produces logits

\[
[-0.1091,\,-4.3779]
\]

for the two classes, and the cross-entropy loss for the target class 0 is approximately 0.01390. The much larger class-0 logit means that the trained victim model classifies this target strongly as class 0.

Using the existing target is important because it connects the gradient diagnostic directly to the GGSS-R reconstruction experiment rather than introducing a new target. The exact target gradient is retained so that the next cell can compare it against the population using the full gradient.


In [26]:
# ============================================================
# CELL 24 — OPTIONAL EXISTING TARGET
# ============================================================

EXISTING_TARGET = (
    REPRO_ROOT /
    "data" /
    "samples" /
    "celeba_target" /
    "target_0000.png"
)

TARGET_METADATA = (
    REPRO_ROOT /
    "data" /
    "target_metadata.pt"
)

target_tensor = None
target_gradient = None

if EXISTING_TARGET.is_file():

    target_pil = Image.open(
        EXISTING_TARGET
    ).convert("RGB")

    target_tensor = preprocess_pil(
        target_pil
    )

    target_result = compute_gradient(
        target_tensor
    )

    target_gradient = flatten_gradient(
        target_result["gradients"]
    ).cpu()

    print("✓ Existing target loaded")
    print(
        "Target logits:",
        target_result["logits"]
    )
    print(
        "Target loss:",
        target_result["loss"]
    )

else:

    print(
        "Existing target not found."
    )

✓ Existing target loaded
Target logits: [-0.10914069 -4.377861  ]
Target loss: 0.013902610167860985


# Cell 25: Target Gradient Identifiability

This cell performs the central target-specific gradient analysis. For each of the 200 population images, it recomputes the complete 100,729,730-dimensional victim gradient and compares it with the target gradient using Euclidean distance, relative distance, cosine similarity, correlation, and pixel MSE.

The nearest population gradient by exact Euclidean distance has a distance of 1.1422 and a cosine similarity of 0.5689. The next closest distances are already around 1.188–1.239, and the first class-1 image in the sorted list appears at distance 1.2655. Most of the closest matches shown are class 0, which is consistent with the target itself belonging to class 0, but the separation is not perfect.

The important point is that the target gradient is distinguishable from the sampled population of 200 images under the full-gradient metrics. That is, none of the 200 alternative images has an exact full-gradient match. 

However the even more crucial observation is that, the pixel MSE values of these closest gradient matches vary considerably (for example, 0.3996 to 1.1153), so being close in gradient space does not uniquely determine pixel-space similarity. Even the lowest MSE found (0.3996) is substantially large, meaning that the images are semantically different despite producing similar gradients. This shows that two images do not necessarily coincide in the pixel-space just because they are close to each other in the gradient space.

This analysis is stronger than the earlier projected-neighbour analysis because the exact full gradient is recomputed. It should still be interpreted as a population-level diagnostic rather than a proof of uniqueness over the entire CelebA distribution: only 200 alternative images were tested.


In [27]:
# ============================================================
# CELL 25 — TARGET GRADIENT IDENTIFIABILITY
# ============================================================

if target_gradient is not None:

    target_rows = []

    for i in range(n):

        # We only stored compact population embeddings.
        # Therefore the population comparison below uses
        # the random-projection representation.

        cosine = cosine_matrix[
            i,
            0
        ]

        target_rows.append({
            "row": i,
            "dataset_index":
                records_df.loc[
                    i,
                    "dataset_index"
                ],
            "class":
                int(classes[i]),
            "projection_cosine_to_row0":
                cosine,
        })

    # --------------------------------------------------------
    # Exact target-to-population comparison.
    #
    # We recompute the full gradient for each image once.
    # This gives the exact metric, rather than the projection.
    # --------------------------------------------------------

    exact_rows = []

    for counter, (dataset_index, class_label) in enumerate(
        all_indices
    ):

        image = load_celeb_image(
            dataset_index
        )

        x = preprocess_pil(
            image
        )

        result = compute_gradient(
            x
        )

        flat = flatten_gradient(
            result["gradients"]
        ).cpu()

        exact_rows.append({
            "dataset_index":
                dataset_index,

            "class":
                class_label,

            "gradient_distance_to_target":
                gradient_distance(
                    flat,
                    target_gradient,
                ),

            "relative_gradient_distance_to_target":
                relative_gradient_distance(
                    flat,
                    target_gradient,
                ),

            "gradient_cosine_to_target":
                gradient_cosine(
                    flat,
                    target_gradient,
                ),

            "gradient_correlation_to_target":
                pearson_gradient_correlation(
                    flat,
                    target_gradient,
                ),

            "pixel_mse_to_target":
                pixel_mse(
                    x.cpu(),
                    target_tensor,
                ),
        })

        del result
        del flat
        del x
        del image

        model.zero_grad(
            set_to_none=True
        )

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    target_df = pd.DataFrame(
        exact_rows
    )

    target_df["PSNR_to_target"] = (
        target_df["pixel_mse_to_target"]
        .apply(psnr_from_mse)
    )

    target_df.to_csv(
        METRIC_DIR /
        "exact_gradient_distance_to_target.csv",
        index=False,
    )

    print(
        target_df.sort_values(
            "gradient_distance_to_target"
        ).head(20)
    )

else:

    print(
        "Skipping target analysis."
    )

     dataset_index  class  gradient_distance_to_target  \
63           84284      0                     1.142171   
78          118681      0                     1.188310   
96           79987      0                     1.191097   
73            4588      0                     1.193737   
51          162267      0                     1.204746   
99          156029      0                     1.221976   
14          110240      0                     1.224098   
13           71751      0                     1.225847   
75           14855      0                     1.230841   
4             4743      0                     1.239964   
160         161813      1                     1.265511   
79           19708      0                     1.265535   
42           39617      0                     1.292757   
45           34216      0                     1.295098   
89           41267      0                     1.318709   
61           42747      0                     1.333332   
86          11

### Interpretation

The exact target comparison gives a more direct test of target-specific gradient identifiability than the earlier population embedding analysis.  

The most important evidence is that the sampled images that are closest in gradient Euclidean distance to the target image still have very high pixel MSE values. This proves that gradient closeness should not be treated as a one-to-one proxy for visual similarity.

As for uniqueness in gradient itself, the results show that the closest of the 200 sampled alternatives has full-gradient distance 1.1422 and cosine 0.5689. This means that the target is not reproduced by an obviously identical gradient among the sampled alternatives. The main caveat here is population coverage. The analysis compares the target against 200 sampled CelebA images, not against every possible input. The result supports gradient distinguishability within only this tested population rather than mathematical uniqueness of the target gradient.


# Cell 26: Layer-Wise Target Gradient Analysis

This cell repeats the target comparison separately for each model parameter tensor. Instead of treating the gradient as one 100-million-dimensional vector, it asks which layers contribute most strongly to the difference between the target and the population.

The mean layer-wise results are:

| Layer | Gradient distance | Relative distance | Cosine | Correlation |
|---|---:|---:|---:|---:|
| `fc1.bias` | 0.0291 | 1.4553 | 0.4341 | 0.4382 |
| `fc1.weight` | 7.7266 | 1.3909 | 0.0503 | 0.0504 |
| `fc2.bias` | 0.1492 | 1.5358 | 0.7228 | 0.7254 |
| `fc2.weight` | 54.5353 | 1.3362 | 0.3366 | 0.3365 |
| `fc3.bias` | 0.7927 | 1.6043 | 1.0000 | 1.0000 |
| `fc3.weight` | 59.1325 | 1.2805 | 0.6024 | 0.6024 |

The raw distance is largest for `fc3.weight`, but raw distance alone is not enough because the layers have very different numbers of parameters and gradient scales. The relative-distance and cosine values show a different picture: `fc3.weight` has the strongest mean cosine among the weight layers, while `fc1.weight` has almost no directional agreement with the target.

This suggests that target-specific variation is not distributed uniformly across the network. However, these are population averages, so they do not establish that one layer alone determines reconstruction quality. The layer-wise result is better viewed as evidence about where the gradient signal differs across images.


In [28]:
# ============================================================
# CELL 26 — LAYER-WISE TARGET GRADIENT ANALYSIS
# ============================================================

if target_gradient is not None:

    # Flatten target layer gradients.
    target_result = compute_gradient(
        target_tensor
    )

    target_layer_grads = [
        g.detach().cpu().reshape(-1)
        for g in target_result["gradients"]
    ]

    layer_rows = []

    for dataset_index, class_label in all_indices:

        image = load_celeb_image(
            dataset_index
        )

        x = preprocess_pil(
            image
        )

        result = compute_gradient(
            x
        )

        for info, g, gt in zip(
            PARAM_INFO,
            result["gradients"],
            target_layer_grads,
        ):

            g = g.detach().cpu().reshape(-1)

            layer_rows.append({
                "dataset_index":
                    dataset_index,

                "class":
                    class_label,

                "layer":
                    info["name"],

                "gradient_distance":
                    gradient_distance(
                        g,
                        gt,
                    ),

                "relative_distance":
                    relative_gradient_distance(
                        g,
                        gt,
                    ),

                "cosine":
                    gradient_cosine(
                        g,
                        gt,
                    ),

                "correlation":
                    pearson_gradient_correlation(
                        g,
                        gt,
                    ),
            })

        del result
        del x
        del image

        model.zero_grad(
            set_to_none=True
        )

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    layer_df = pd.DataFrame(
        layer_rows
    )

    layer_df.to_csv(
        METRIC_DIR /
        "layerwise_gradient_target_comparison.csv",
        index=False,
    )

    print(
        layer_df.groupby("layer")[
            [
                "gradient_distance",
                "relative_distance",
                "cosine",
                "correlation",
            ]
        ].mean()
    )

            gradient_distance  relative_distance    cosine  correlation
layer                                                                  
fc1.bias             0.029051           1.455310  0.434108     0.438153
fc1.weight           7.726555           1.390879  0.050270     0.050419
fc2.bias             0.149223           1.535812  0.722787     0.725385
fc2.weight          54.535273           1.336190  0.336622     0.336536
fc3.bias             0.792682           1.604330  1.000000     1.000000
fc3.weight          59.132462           1.280507  0.602432     0.602432


### Interpretation

The layer-wise results show that the gradient signal is distributed very unevenly across the MLP. The largest raw mean distances occur in `fc3.weight` and `fc2.weight`, but those layers also have different parameter counts and gradient scales, so raw distance should not be compared across layers without normalization.

The relative-distance and cosine columns give additional context. `fc1.weight` has a very low mean cosine of 0.0503, whereas `fc3.weight` has a mean cosine of 0.6024. The bias terms behave differently again, with `fc3.bias` having cosine 1.0 because the corresponding two-dimensional gradient is highly constrained.

The practical conclusion is not that one particular layer is solely responsible for identifiability. Rather, different parameter groups carry different amounts and types of target-dependent variation, which is relevant when considering how the leaked gradient constrains the input.


# Cell 27 — Load Existing GGSS-R Reconstructions

This cell loads the saved \(x_0\) reconstruction snapshots from the original GGSS-R experiment. Ten files are found, corresponding to reconstruction states `x_0000.png` through `x_0900.png`.

These files are not regenerated here. They are read from the existing reproduction directory so that the diagnostic can ask a separate question: as the GGSS-R reconstruction evolves, does its gradient move toward the target gradient at the same time that the image becomes closer to the target?

Because the files come from the original experiment, this cell also provides the bridge between the new gradient-identifiability analysis and the actual reconstruction trajectory.


In [29]:
# ============================================================
# CELL 27 — LOAD EXISTING GGSS-R RECONSTRUCTIONS
# ============================================================

X0_DIR = (
    REPRO_ROOT /
    "results" /
    "baseline_fixed_mr_v3" /
    "reconstruction" /
    "progresss" /
    "00000" /
    "x0"
)

reconstruction_files = sorted(
    X0_DIR.glob("x_*.png")
)

print(
    "Found",
    len(reconstruction_files),
    "saved x0 reconstructions."
)

for p in reconstruction_files:
    print(" ", p.name)

Found 10 saved x0 reconstructions.
  x_0000.png
  x_0100.png
  x_0200.png
  x_0300.png
  x_0400.png
  x_0500.png
  x_0600.png
  x_0700.png
  x_0800.png
  x_0900.png


# Cell 28 — Reconstruction vs Gradient Population

This cell evaluates every saved GGSS-R reconstruction against the target in both image space and gradient space. For each reconstruction it computes pixel MSE, PSNR, LPIPS, exact full-gradient distance, relative gradient distance, gradient cosine, and gradient correlation.

The results show a clear progression. From `x_0000` to `x_0900`, pixel MSE increases from 0.6085 to 0.8997, PSNR decreases from 2.158 dB to 0.459 dB, and LPIPS increases from 0.7490 to 0.8614. At the same time, the exact gradient distance increases from 0.2898 to 0.8918 and gradient cosine decreases from 0.9764 to 0.7454.

Thus, in this saved reconstruction trajectory, the reconstructions that are closer to the target in image space are also closer to the target in gradient space. The relationship is especially visible in the first several snapshots: gradient cosine falls steadily from 0.9764 to 0.9608 while the image metrics also move away from the target.

There is an important caveat in the nearest-population columns. The exact target-relative gradient metrics use the full gradient, but the `nearest_population_*` identity is obtained from the 256-dimensional random projection because the complete population gradients were not retained. Therefore, those nearest-population identities should not be treated as exact full-gradient nearest neighbours.

The reconstruction trajectory therefore provides useful evidence for the proposed link between gradient proximity and reconstruction quality, but it does not by itself establish causality. Both quantities are measured along the same reconstruction trajectory, and other properties of the optimization process may contribute to the observed trend.


In [30]:
# ============================================================
# CELL 28 — RECONSTRUCTION vs GRADIENT POPULATION
# ============================================================

if target_gradient is not None:

    recon_rows = []

    for path in reconstruction_files:

        pil = Image.open(
            path
        ).convert("RGB")

        x = preprocess_pil(
            pil
        )

        result = compute_gradient(
            x
        )

        flat = flatten_gradient(
            result["gradients"]
        ).cpu()

        mse = pixel_mse(
            x.cpu(),
            target_tensor,
        )

        lp = compute_lpips(
            x,
            target_tensor,
        )

        # Find nearest population gradient using
        # exact target-relative metrics.
        #
        # For population-wide nearest-neighbour identity,
        # use the random projection because we don't retain
        # all full gradients.

        embedding = project_gradient(
            flat
        )

        emb_norm = (
            np.linalg.norm(embedding)
            + 1e-12
        )

        pop_cosines = (
            embeddings
            @
            embedding
            /
            (
                np.linalg.norm(
                    embeddings,
                    axis=1,
                )
                *
                emb_norm
                +
                1e-12
            )
        )

        nearest = np.argsort(
            -pop_cosines
        )[:K_NEIGHBORS]

        recon_rows.append({
            "reconstruction":
                path.name,

            "pixel_mse_to_target":
                mse,

            "PSNR_to_target":
                psnr_from_mse(mse),

            "LPIPS_to_target":
                lp,

            "gradient_distance_to_target":
                gradient_distance(
                    flat,
                    target_gradient,
                ),

            "relative_gradient_distance":
                relative_gradient_distance(
                    flat,
                    target_gradient,
                ),

            "gradient_cosine_to_target":
                gradient_cosine(
                    flat,
                    target_gradient,
                ),

            "gradient_correlation_to_target":
                pearson_gradient_correlation(
                    flat,
                    target_gradient,
                ),

            "nearest_population_row":
                int(nearest[0]),

            "nearest_population_dataset_index":
                int(
                    records_df.loc[
                        nearest[0],
                        "dataset_index"
                    ]
                ),

            "nearest_population_class":
                int(
                    records_df.loc[
                        nearest[0],
                        "class"
                    ]
                ),

            "nearest_population_gradient_cosine":
                float(
                    pop_cosines[nearest[0]]
                ),
        })

        del result
        del flat
        del x
        del pil

        model.zero_grad(
            set_to_none=True
        )

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    recon_df = pd.DataFrame(
        recon_rows
    )

    recon_df.to_csv(
        METRIC_DIR /
        "reconstruction_gradient_population_analysis.csv",
        index=False,
    )

    print(recon_df)

  reconstruction  pixel_mse_to_target  PSNR_to_target  LPIPS_to_target  \
0     x_0000.png             0.608466        2.157634         0.748973   
1     x_0100.png             0.636612        1.961249         0.754557   
2     x_0200.png             0.665332        1.769613         0.756987   
3     x_0300.png             0.691170        1.604154         0.765866   
4     x_0400.png             0.718710        1.434464         0.769728   
5     x_0500.png             0.743560        1.286839         0.781476   
6     x_0600.png             0.773772        1.113867         0.781508   
7     x_0700.png             0.809312        0.918838         0.796370   
8     x_0800.png             0.828239        0.818442         0.832051   
9     x_0900.png             0.899684        0.459102         0.861422   

   gradient_distance_to_target  relative_gradient_distance  \
0                     0.289815                    0.227284   
1                     0.309412                    0.257449   

### Interpretation

The reconstruction trajectory gives the clearest direct connection between image quality and target-gradient similarity in this notebook. As the saved snapshots progress from `x_0000` to `x_0900`, image quality consistently worsens according to all three image metrics: MSE rises from 0.6085 to 0.8997, PSNR falls from 2.16 dB to 0.46 dB, and LPIPS rises from 0.7490 to 0.8614.

The gradient metrics move in the same direction. Exact gradient distance rises from 0.2898 to 0.8918, while cosine similarity falls from 0.9764 to 0.7454. Thus, the reconstruction that is best among these saved snapshots is also the one whose victim gradient is closest to the target gradient.

This alignment is consistent with the hypothesis that matching the victim gradient is informative about the target image. It is not, however, a causal proof: the snapshots are successive states of one reconstruction process, so image quality and gradient similarity are changing together rather than being independently manipulated. The nearest-population identity is also based on the compact projection, not the full population gradients, and should be interpreted accordingly.


# Cell 29 — Visualizations

This cell saves four plots that summarize the main population-level relationships studied in the notebook:

1. PCA of the projected victim-gradient population, with the two CelebA classes shown separately.
2. The distribution of pairwise gradient cosine similarity for same-class and different-class pairs.
3. Gradient embedding distance versus pixel MSE.
4. Gradient cosine similarity versus pixel MSE.

These plots are intended to make the numerical findings easier to inspect. In particular, the cosine-versus-MSE plot should reflect the relatively strong negative Spearman correlation of \(-0.6348\), while the gradient-distance-versus-MSE plot should show the much weaker relationship corresponding to \(
ho=0.1250\).

The output confirms that all plots were saved to the diagnostic `plots` directory. The figures summarize the compact 256-dimensional population analysis; they should therefore be interpreted separately from the exact full-gradient target analysis.


In [31]:
# ============================================================
# CELL 29 — VISUALIZATIONS
# ============================================================

import matplotlib.pyplot as plt

# ------------------------------------------------------------
# PCA
# ------------------------------------------------------------

plt.figure(
    figsize=(8, 6)
)

for c in [0, 1]:

    mask = classes == c

    plt.scatter(
        embedding_pca[mask, 0],
        embedding_pca[mask, 1],
        label=f"class {c}",
        alpha=0.65,
    )

plt.xlabel("Gradient PC1")
plt.ylabel("Gradient PC2")
plt.title(
    "Victim-gradient population in PCA space"
)
plt.legend()
plt.tight_layout()

plt.savefig(
    PLOT_DIR /
    "gradient_pca_class_separation.png",
    dpi=200,
)

plt.close()

# ------------------------------------------------------------
# Gradient cosine distribution
# ------------------------------------------------------------

plt.figure(
    figsize=(8, 6)
)

plt.hist(
    same_class_cosines,
    bins=40,
    alpha=0.6,
    label="same class",
)

plt.hist(
    different_class_cosines,
    bins=40,
    alpha=0.6,
    label="different class",
)

plt.xlabel(
    "Gradient cosine similarity"
)

plt.ylabel(
    "Number of pairs"
)

plt.title(
    "Pairwise victim-gradient similarity"
)

plt.legend()
plt.tight_layout()

plt.savefig(
    PLOT_DIR /
    "gradient_cosine_distribution.png",
    dpi=200,
)

plt.close()

# ------------------------------------------------------------
# Gradient vs pixel distance
# ------------------------------------------------------------

plt.figure(
    figsize=(8, 6)
)

plt.scatter(
    relationship_df[
        "gradient_distance"
    ],
    relationship_df[
        "pixel_mse"
    ],
    s=5,
    alpha=0.25,
)

plt.xlabel(
    "Gradient embedding distance"
)

plt.ylabel(
    "Pixel-space MSE"
)

plt.title(
    "Does gradient proximity imply image proximity?"
)

plt.tight_layout()

plt.savefig(
    PLOT_DIR /
    "gradient_distance_vs_pixel_mse.png",
    dpi=200,
)

plt.close()

# ------------------------------------------------------------
# Gradient cosine vs pixel MSE
# ------------------------------------------------------------

plt.figure(
    figsize=(8, 6)
)

plt.scatter(
    relationship_df[
        "gradient_cosine"
    ],
    relationship_df[
        "pixel_mse"
    ],
    s=5,
    alpha=0.25,
)

plt.xlabel(
    "Gradient cosine similarity"
)

plt.ylabel(
    "Pixel-space MSE"
)

plt.title(
    "Gradient similarity vs image similarity"
)

plt.tight_layout()

plt.savefig(
    PLOT_DIR /
    "gradient_cosine_vs_pixel_mse.png",
    dpi=200,
)

plt.close()

print(
    "✓ Plots saved to:",
    PLOT_DIR
)

✓ Plots saved to: /content/drive/MyDrive/diagnostics_14th_august/plots


### Interpretation

The saved figures provide visual checks of the numerical findings. The PCA plot shows the low-dimensional structure of the projected population, the cosine histogram compares within-class and across-class similarity, and the two scatter plots show how gradient similarity relates to pixel-space MSE.

The most important expected visual distinction is between the two gradient/image plots: the cosine-versus-MSE relationship should show a much clearer trend than the raw gradient-distance-versus-MSE relationship, matching the Spearman coefficients of -0.6348 and 0.1250 respectively.


# Cell 30 — Experiment Summary

This cell writes a compact JSON record containing the configuration and provenance of the experiment. It records the random seed, device, checkpoint, model identity, parameter count, population sizes, projection settings, target class, whether the existing target was used, and the number of reconstruction snapshots found.

The final output confirms the main experimental conditions: the diagnostic used the `MLP_1 / paper MLP-3` model with 100,729,730 parameters, 100 images from each class, a 256-dimensional projection with 4,096 coordinates per dimension, and 100,000 sampled gradient coordinates. The existing target and all 10 saved reconstruction snapshots were available.

The summary also confirms the file-separation requirement: all diagnostic outputs were written under `diagnostics_14th_august`, and nothing was written into the original `GGSS_R_unperturbed_reproduction` tree.

Taken together, the experiment finds three relevant patterns. First, the projected gradient population contains measurable class structure but substantial overlap. Second, gradient similarity—especially cosine similarity—is meaningfully related to image similarity in the sampled population. Third, along the existing GGSS-R reconstruction trajectory, improved image similarity occurs together with improved target-gradient similarity. These observations support the narrower hypothesis that the trained victim gradient contains image-specific information relevant to reconstruction, while the population size, random projection, and observational nature of the experiment limit how broadly that conclusion can be generalized.


In [32]:
# ============================================================
# CELL 30 — EXPERIMENT SUMMARY
# ============================================================

summary = {
    "seed": SEED,

    "device": str(DEVICE),

    "checkpoint": str(
        CHECKPOINT_PATH
    ),

    "checkpoint_modified": (
        CHECKPOINT_PATH.stat()
        .st_mtime
    ),

    "model": "MLP_1 / paper MLP-3",

    "parameter_count":
        num_params,

    "n_class0":
        N_CLASS0,

    "n_class1":
        N_CLASS1,

    "projection_dimension":
        PROJECTION_DIM,

    "per_projection_coordinates":
        PROJECTION_COORDS_PER_DIM,

    "per_element_sample_count":
        len(sampled_global_coords),

    "target_class":
        TARGET_CLASS,

    "existing_target_used":
        target_gradient is not None,

    "existing_reconstructions_found":
        len(reconstruction_files),

    "diagnostic_root":
        str(DIAG_ROOT),
}

with open(
    REPORT_DIR /
    "experiment_summary.json",
    "w",
) as f:

    json.dump(
        summary,
        f,
        indent=2,
    )

print("=" * 70)
print("DIAGNOSTIC COMPLETE")
print("=" * 70)

for key, value in summary.items():

    print(
        f"{key}: {value}"
    )

print()
print(
    "ALL OUTPUTS ARE IN:"
)

print(
    DIAG_ROOT
)

print()
print(
    "Nothing was written into:"
)

print(
    REPRO_ROOT
)

DIAGNOSTIC COMPLETE
seed: 2026
device: cuda:0
checkpoint: /content/drive/MyDrive/GGSS_R_unperturbed_reproduction/model_state/MLP_1.pth
checkpoint_modified: 1786604179.0
model: MLP_1 / paper MLP-3
parameter_count: 100729730
n_class0: 100
n_class1: 100
projection_dimension: 256
per_projection_coordinates: 4096
per_element_sample_count: 100000
target_class: 0
existing_target_used: True
existing_reconstructions_found: 10
diagnostic_root: /content/drive/MyDrive/diagnostics_14th_august

ALL OUTPUTS ARE IN:
/content/drive/MyDrive/diagnostics_14th_august

Nothing was written into:
/content/drive/MyDrive/GGSS_R_unperturbed_reproduction


### Interpretation

The final configuration confirms that the experiment was run with a fixed seed of 2026 on the `MLP_1 / paper MLP-3` checkpoint, using 200 population images and the compact gradient representations described above. The target and all ten saved GGSS-R reconstruction snapshots were available, so the target-specific and reconstruction-trajectory analyses were both executed.

The combined evidence is consistent with a narrower form of the gradient-identifiability hypothesis: the trained victim gradients contain information that varies with the input image, and that information is measurably related to reconstruction quality. The evidence is strongest in the target/reconstruction analysis, where better saved reconstructions have gradients closer to the target gradient.

There are three important limits to the conclusion. The population contains only 200 sampled alternatives; most population-wide analyses use a 256-dimensional random projection rather than the complete gradient; and the reconstruction comparison follows an existing optimization trajectory rather than manipulating gradient similarity independently. These limits should be kept explicit when using this diagnostic to explain why GGSS-R can reconstruct information from the victim gradient.
